In [1]:
#fill NA with None
from sklearn import tree
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import random
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold
from statistics import mean
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.naive_bayes import CategoricalNB

In [2]:
#https://stackoverflow.com/questions/20224526/how-to-extract-the-decision-rules-from-scikit-learn-decision-tree
def get_code(tree, feature_names):
        left      = tree.tree_.children_left
        right     = tree.tree_.children_right
        threshold = tree.tree_.threshold
        features  = [feature_names[i] for i in tree.tree_.feature]
        value = tree.tree_.value

        def recurse(left, right, threshold, features, node):
                if (threshold[node] != -2):
                        print("if ( " + features[node] + " <= " + str(threshold[node]) + " ) {")
                        if left[node] != -1:
                                recurse (left, right, threshold, features,left[node])
                        print( "} else {")
                        if right[node] != -1:
                                recurse (left, right, threshold, features,right[node])
                        print( "}")
                else:
                        print( "return " + str(value[node]))

        recurse(left, right, threshold, features, 0)

In [3]:
# data = pd.read_excel('dataset_all_original.xlsx') 
# data = pd.read_excel('dataset_all_gen_ssdp.xlsx') 
# data = pd.read_excel('dataset_all_gen.xlsx') 
data = pd.read_excel('dataset_all_original_ssdp.xlsx') 
# data = pd.read_excel('dataset_all_original_add_nobias.xlsx')
dataset_all_original = pd.read_excel('dataset_all_original.xlsx')
dataset_jaccard_original = pd.read_excel('dataset_jaccard_original.xlsx')
dataset_myroom_original = pd.read_excel('dataset_myroom_original.xlsx')
dataset_university_original = pd.read_excel('dataset_university_original.xlsx')

In [4]:
data = data.sample(frac = 1) 
dataset_all_original = dataset_all_original.sample(frac = 1) 
dataset_jaccard_original = dataset_jaccard_original.sample(frac = 1) 
dataset_myroom_original = dataset_myroom_original.sample(frac = 1) 
dataset_university_original = dataset_university_original.sample(frac = 1) 

In [5]:

data = data.reset_index(drop=True)
dataset_all_original = dataset_all_original.reset_index(drop=True)
dataset_jaccard_original = dataset_jaccard_original.reset_index(drop=True)
dataset_myroom_original = dataset_myroom_original.reset_index(drop=True)
dataset_university_original = dataset_university_original.reset_index(drop=True)

In [6]:
data.isnull().sum()

type             0
name             0
vendor           0
DNS           2491
mDNS           937
nupnp        11636
open_port      875
ssdp_type     1361
ssdp_t        1361
ssdp_v        1361
dtype: int64

In [7]:
data = data.fillna("NaN")

In [8]:
#fill Na
dataset_all_original = dataset_all_original.fillna("NaN")
dataset_jaccard_original = dataset_jaccard_original.fillna("NaN")
dataset_myroom_original = dataset_myroom_original.fillna("NaN")
dataset_university_original = dataset_university_original.fillna("NaN")

In [9]:
data.isnull().sum()

type         0
name         0
vendor       0
DNS          0
mDNS         0
nupnp        0
open_port    0
ssdp_type    0
ssdp_t       0
ssdp_v       0
dtype: int64

In [10]:
data.dtypes

type         object
name         object
vendor       object
DNS          object
mDNS         object
nupnp        object
open_port    object
ssdp_type    object
ssdp_t       object
ssdp_v       object
dtype: object

In [11]:
data = data.astype('category')

dataset_all_original = dataset_all_original.astype('category')

dataset_jaccard_original = dataset_jaccard_original.astype('category')

dataset_myroom_original = dataset_myroom_original.astype('category')

dataset_university_original = dataset_university_original.astype('category')

In [12]:
data.dtypes

type         category
name         category
vendor       category
DNS          category
mDNS         category
nupnp        category
open_port    category
ssdp_type    category
ssdp_t       category
ssdp_v       category
dtype: object

In [13]:
data

,type,name,vendor,DNS,mDNS,nupnp,open_port,ssdp_type,ssdp_t,ssdp_v
0,Hubs,Amazon Echo,Amazon Technologies Inc.,"('device-metrics-us.amazon.com', 'A')",NaN,NaN,tcp/80,NaN,NaN,NaN
1,Health-Monitor,Withings Smart Monitor,Withings,yellowbook.com,NaN,NaN,tcp/22,NOTIFY * HTTP/1.1,hue-bridgeid,001788FFFE6EBD93
2,Energy Management,Belkin wemo motion sensor,Belkin International Inc.,rakuten.co.jp,_tempsoft._udp.local.,NaN,tcp/1900,NOTIFY * HTTP/1.1,Location,http://192.168.1.141:8060/
3,Hubs,Caseta Wireless Hub,"Lutron Electronics Co., Inc",slashdot.org,_fix san._udp.local.,NaN,tcp/22,NOTIFY * HTTP/1.1,NT,urn:schemas-upnp-org:device:WANDevice:1
4,Cameras,NestCam,Nest Labs Inc.,seattletimes.com,_hatity._tcp.local.,NaN,tcp/1883,NOTIFY * HTTP/1.1,OPT,http://schemas.upnp.org/upnp/1/0/;
5,Appliances,iRobot,iRobot Corporation,upenn.edu,_bamity._udp.local.,NaN,tcp/443,NOTIFY * HTTP/1.1,LOCATION,http://192.168.0.102:80/description.xml
6,Cameras,Netgear Arlo Camera,Arlo,studiopress.com,_lotstring._tcp.local.,NaN,tcp/443,NOTIFY * HTTP/1.1,LOCATION,http://192.168.0.1:1900/igd.xml
7,Health-Monitor,NEST Protect smoke alarm,Nest Labs Inc.,photobucket.com,_stronghold._udp.local.,NaN,tcp/1883,NOTIFY * HTTP/1.1,Server,SDK 4.3.0.0 UPnP/1.0 MiniUPnPd/1.6
8,Health-Monitor,NEST Protect smoke alarm,Nest Labs Inc.,netscape.com,_solarbreeze._tcp.local.,NaN,tcp/22,NOTIFY * HTTP/1.1,NT,urn:dial-multiscreen-org:service:dial:1
9,Appliances,Triby Speaker Smart Speaker,Invoxia,reuters.com,_zaam-dox._udp.local.,NaN,tcp/80,NOTIFY * HTTP/1.1,MX,1


In [14]:
y = data.drop_duplicates(keep='first')
# y = y.drop(['name'], axis=1)
# y = y.drop(['Name'], axis=1)
# y = y.drop_duplicates(keep='first')
# y = y.reset_index()
# shuffle the DataFrame rows
# y = y.sample(frac = 1) 


In [15]:
#drop_duplicates

dataset_all_original_y = dataset_all_original.drop_duplicates(keep='first')

dataset_jaccard_original_y = dataset_jaccard_original.drop_duplicates(keep='first')

dataset_myroom_original_y = dataset_myroom_original.drop_duplicates(keep='first')

dataset_university_original_y = dataset_university_original.drop_duplicates(keep='first')

In [16]:
# #drop name
# # dataset_all_gen_y = dataset_all_gen_y.drop(['Name'], axis=1)
# dataset_all_original_y = dataset_all_original_y.drop(['name'], axis=1)
# # dataset_jaccard_gen_y = dataset_jaccard_gen_y.drop(['Name'], axis=1)
# dataset_jaccard_original_y = dataset_jaccard_original_y.drop(['name'], axis=1)
# # dataset_myroom_gen_y = dataset_myroom_gen_y.drop(['Name'], axis=1)
# dataset_myroom_original_y = dataset_myroom_original_y.drop(['name'], axis=1)
# # dataset_university_gen_y = dataset_university_gen_y.drop(['Name'], axis=1)
# dataset_university_original_y = dataset_university_original_y.drop(['name'], axis=1)

In [17]:
# #drop_duplicates

# dataset_all_original_y = dataset_all_original_y.drop_duplicates(keep='first')

# dataset_jaccard_original_y = dataset_jaccard_original_y.drop_duplicates(keep='first')

# dataset_myroom_original_y = dataset_myroom_original_y.drop_duplicates(keep='first')

# dataset_university_original_y = dataset_university_original_y.drop_duplicates(keep='first')

In [18]:
y = y.astype(str)
len(y)

12000

In [19]:
y = y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
# y = y.apply(LabelEncoder().fit_transform)

In [20]:

dataset_all_original_y = dataset_all_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')

dataset_jaccard_original_y = dataset_jaccard_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')

dataset_myroom_original_y = dataset_myroom_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')

dataset_university_original_y = dataset_university_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')

In [21]:
# dataset_all_original_y

In [22]:
list_data = []
# list_data.append(y)
# # list_data.append(dataset_all_gen_y)
list_data.append(dataset_all_original_y)
# # list_data.append(dataset_jaccard_gen_y)
# list_data.append(dataset_jaccard_original_y)
# # list_data.append(dataset_myroom_gen_y)
# list_data.append(dataset_myroom_original_y)
# # list_data.append(dataset_university_gen_y)
# list_data.append(dataset_university_original_y)
len(list_data)

1

In [23]:
accuracy = []
accuracy1 = []

std_ = []

In [24]:
for data in list_data:
    acc = []
    accuracy_t = []
    for i in range(10):
        accuracy_temp = []
        kf = KFold(n_splits=10, random_state=None, shuffle=True)
#         kf.get_n_splits(data)
        print(i)
        for train_index, test_index in kf.split(data):
# #             print(index)
#             print("TRAIN:", train_index, "TEST:", test_index)
            train, test = data.iloc[train_index], data.iloc[test_index]
#             print("TRAIN:", train, "TEST:", test)
        #     print(len(train))
#             print("TRAIN:", len(train), "TEST:", len(test))
            label_data = train.type
            train_data = train.drop(['type'], axis=1)
            tlabel_data = test.type
            test_data = test.drop(['type'], axis=1)
            truelabel_data = pd.DataFrame(tlabel_data)
            truelabel_data = truelabel_data.reset_index(drop=True)
#             clf = RandomForestClassifier()
#             clf = ExtraTreesClassifier()
#             clf = tree.DecisionTreeClassifier()
            clf = CategoricalNB()
            clf = clf.fit(train_data,label_data )
            predict = clf.predict(test_data)
            result_label_data = pd.DataFrame(columns = ['type'])
            result_label_data.type = predict
            compare = pd.DataFrame(columns = ['truelabel_data','predictedlabel'])
            compare.truelabel_data = truelabel_data.type
            compare.predictedlabel = result_label_data.type
#             print(compare)
            t = (accuracy_score(compare.truelabel_data, compare.predictedlabel))
            accuracy_temp.append(t)
            acc.append(t)
#             print(i,"  Accuracy: ",t)
        accuracy_t.append(mean(accuracy_temp))
    std_.append(np.std(acc))
    accuracy.append(mean(acc))
#     accuracy1.append(mean(acc))  
#     if(mean(accuracy_t)!=mean(acc)):
#         print("False")

0


IndexError: index 21 is out of bounds for axis 1 with size 21

In [ ]:
print(len(y),"\t Accuracy for dataset_gen: \t", accuracy[0])
# print(len(dataset_all_gen_y),"\t Accuracy for dataset_all_gen: \t", accuracy[1])
# print(len(dataset_all_original_y),"\t Accuracy for dataset_all_original: \t", accuracy[1])
# print(len(dataset_jaccard_gen_y),"\t Accuracy for dataset_jaccard_gen: \t", accuracy[3])
# print(len(dataset_jaccard_original_y),"\t Accuracy for dataset_jaccard_original: \t", accuracy[2])
# # print(len(dataset_myroom_gen_y),"\t Accuracy for dataset_myroom_gen: \t", accuracy[5])
# print(len(dataset_myroom_original_y),"\t Accuracy for dataset_myroom_original: \t", accuracy[3])
# # print(len(dataset_university_gen_y),"\t Accuracy for dataset_university_gen: \t", accuracy[7])
# print(len(dataset_university_original_y),"\t Accuracy for dataset_university_original: \t", accuracy[4])
print(len(y),"\t STD for dataset_gen: \t", std_[0])
# print(len(dataset_all_gen_y),"\t STD for dataset_all_gen: \t", std_[1])
# print(len(dataset_all_original_y),"\t STD for dataset_all_original: \t", std_[1])
# # print(len(dataset_jaccard_gen_y),"\t STD for dataset_jaccard_gen: \t", std_[3])
# print(len(dataset_jaccard_original_y),"\t STD for dataset_jaccard_original: \t", std_[2])
# # print(len(dataset_myroom_gen_y),"\t STD for dataset_myroom_gen: \t", std_[5])
# print(len(dataset_myroom_original_y),"\t STD for dataset_myroom_original: \t", std_[3])
# # print(len(dataset_university_gen_y),"\t STD for dataset_university_gen: \t", std_[7])
# print(len(dataset_university_original_y),"\t STD for dataset_university_original: \t", std_[4])



In [ ]:
print(len(y),"\t Accuracy for dataset_gen: \t", accuracy[0])
# print(len(dataset_all_gen_y),"\t Accuracy for dataset_all_gen: \t", accuracy[1])
print(len(dataset_all_original_y),"\t Accuracy for dataset_all_original: \t", accuracy[1])
# print(len(dataset_jaccard_gen_y),"\t Accuracy for dataset_jaccard_gen: \t", accuracy[3])
print(len(dataset_jaccard_original_y),"\t Accuracy for dataset_jaccard_original: \t", accuracy[2])
# print(len(dataset_myroom_gen_y),"\t Accuracy for dataset_myroom_gen: \t", accuracy[5])
print(len(dataset_myroom_original_y),"\t Accuracy for dataset_myroom_original: \t", accuracy[3])
# print(len(dataset_university_gen_y),"\t Accuracy for dataset_university_gen: \t", accuracy[7])
print(len(dataset_university_original_y),"\t Accuracy for dataset_university_original: \t", accuracy[4])

In [ ]:
print(len(y),"\t STD for dataset_gen: \t", std_[0])
# print(len(dataset_all_gen_y),"\t STD for dataset_all_gen: \t", std_[1])
print(len(dataset_all_original_y),"\t STD for dataset_all_original: \t", std_[1])
# print(len(dataset_jaccard_gen_y),"\t STD for dataset_jaccard_gen: \t", std_[3])
print(len(dataset_jaccard_original_y),"\t STD for dataset_jaccard_original: \t", std_[2])
# print(len(dataset_myroom_gen_y),"\t STD for dataset_myroom_gen: \t", std_[5])
print(len(dataset_myroom_original_y),"\t STD for dataset_myroom_original: \t", std_[3])
# print(len(dataset_university_gen_y),"\t STD for dataset_university_gen: \t", std_[7])
print(len(dataset_university_original_y),"\t STD for dataset_university_original: \t", std_[4])

